In [ ]:
import json
import os

# --- AYARLAR ---
# Eski sunucudan gelen JSON dosyası
INPUT_JSON = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/lit_pcba_paths_all_pdbs.json"
# Yeni oluşturulacak JSON dosyası
OUTPUT_JSON = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/lit_pcba_paths_prepared_arf.json"

# Şu anki klasör (LIT-PCBA klasörü)
# NOT: Scripti LIT-PCBA klasörünün içinde çalıştırdığını varsayıyorum.
BASE_DIR = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA"

def main():
    print(f"📂 Okunuyor: {INPUT_JSON}")
    if not os.path.exists(INPUT_JSON):
        print(f"❌ HATA: {INPUT_JSON} bulunamadı!")
        return

    with open(INPUT_JSON, 'r') as f:
        data = json.load(f)

    new_data = {}
    print("🔄 Yollar güncelleniyor...")
    
    # LIT-PCBA klasör yapısı genelde şöyledir:
    # LIT-PCBA/
    #   ADRB2/
    #     actives.sdf (veya .gz)
    #     decoys.sdf (veya .gz)
    #     3p0g_protein.mol2
    #     3p0g_ligand.mol2
    
    for target_name, records in data.items():
        new_records = []
        target_folder = os.path.join(BASE_DIR, target_name)
        
        # Her hedef (Target) için Active ve Decoy dosyalarını bulalım
        # Genelde isimleri "actives.sdf" ve "decoys.sdf" olur.
        # Bazen .gz sıkıştırmalı olabilir.
        actives_path = os.path.join(target_folder, "actives.sdf")
        decoys_path = os.path.join(target_folder, "decoys.sdf")
        
        # Eğer .gz versiyonları varsa onları tercih et (yoksa normalini al)
        if not os.path.exists(actives_path) and os.path.exists(actives_path + ".gz"):
            actives_path += ".gz"
        if not os.path.exists(decoys_path) and os.path.exists(decoys_path + ".gz"):
            decoys_path += ".gz"

        for rec in records:
            # Eski yollardan sadece dosya isimlerini al
            # Örn: .../ADRB2/3p0g_protein.mol2 -> 3p0g_protein.mol2
            prot_name = os.path.basename(rec['protein_mol2'])
            lig_name = os.path.basename(rec['ligand_mol2'])
            
            # Yeni yolları oluştur
            new_prot_path = os.path.join(target_folder, prot_name)
            new_lig_path = os.path.join(target_folder, lig_name)
            
            # Bazen LIT-PCBA'da protein dosyası .pdb olabilir, kontrol edelim
            if not os.path.exists(new_prot_path):
                # .mol2 yoksa .pdb dene
                pdb_ver = new_prot_path.replace(".mol2", ".pdb")
                if os.path.exists(pdb_ver):
                    new_prot_path = pdb_ver

            new_rec = {
                "name": rec['name'],
                "protein_file": new_prot_path, # İsmi protein_file yaptım (pdb veya mol2 olabilir)
                "ligand_file": new_lig_path,
                "actives_file": actives_path,  # Tüm PDB şablonları aynı active/decoy setini kullanır
                "decoys_file": decoys_path
            }
            new_records.append(new_rec)
            
        new_data[target_name] = new_records

    print(f"💾 Kaydediliyor: {OUTPUT_JSON}")
    with open(OUTPUT_JSON, 'w') as f:
        json.dump(new_data, f, indent=4)
        
    print("✅ İşlem Tamamlandı!")
    
    # Kontrol için ilk kaydı basalım
    first_key = list(new_data.keys())[0]
    print(f"\nÖrnek Kayıt ({first_key}):")
    print(json.dumps(new_data[first_key][0], indent=2))

if __name__ == "__main__":
    main()

In [ ]:
import json
import os

# Ayarlar
INPUT_JSON = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/lit_pcba_paths_all_pdbs.json" # Orijinal dosya
OUTPUT_JSON = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/lit_pcba_paths_prepared_arf.json" # Çıktı
REAL_BASE_DIR = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA"

def main():
    if not os.path.exists(INPUT_JSON):
        print(f"❌ {INPUT_JSON} bulunamadı!")
        return

    with open(INPUT_JSON, 'r') as f:
        data = json.load(f)

    new_data = {}
    
    for target_name, records in data.items():
        new_records = []
        target_folder = os.path.join(REAL_BASE_DIR, target_name)
        
        # Artık .sdf dosyalarını oluşturduğumuz için bunları gösteriyoruz
        actives_path = os.path.join(target_folder, "actives.sdf")
        decoys_path = os.path.join(target_folder, "decoys.sdf")
        
        for rec in records:
            prot_name = os.path.basename(rec['protein_mol2'])
            lig_name = os.path.basename(rec['ligand_mol2'])
            
            new_rec = {
                "name": rec['name'],
                "protein_file": os.path.join(target_folder, prot_name),
                "ligand_file": os.path.join(target_folder, lig_name),
                "actives_file": actives_path,
                "decoys_file": decoys_path
            }
            new_records.append(new_rec)
            
        new_data[target_name] = new_records

    with open(OUTPUT_JSON, 'w') as f:
        json.dump(new_data, f, indent=4)
        
    print(f"✅ JSON güncellendi: {OUTPUT_JSON}")
    
    # Kontrol
    if "ADRB2" in new_data:
        print(f"Active Path: {new_data['ADRB2'][0]['actives_file']}")

if __name__ == "__main__":
    main()

In [1]:
import pickle
import torch

# ESM'li bir dosya yolu seçelim
esm_file = '/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/planet_esm2_pkl/4tjy_binder_esm512d.pkl'

with open(esm_file, 'rb') as f:
    data = pickle.load(f)

print(f"--- ESM Dosya İncelemesi ---")
print(f"Veri Tipi: {type(data)}")

if isinstance(data, list):
    print(f"Liste Uzunluğu: {len(data)}")
    print(f"İlk Eleman Tipi: {type(data[0])}")
    if hasattr(data[0], 'shape'): print(f"Boyut (Shape): {data[0].shape}")
elif isinstance(data, dict):
    print(f"Sözlük Anahtarları (Keys): {data.keys()}")

ModuleNotFoundError: No module named 'chemutils'

In [ ]:
import os
import numpy as np
from rdkit import Chem

# --- AYARLAR ---
PROTEIN_PDB = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/ADRB2/3p0g_protein.pdb"
LIGAND_SDF = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/ADRB2/actives.sdf"

def get_centroid(mol):
    conf = mol.GetConformer()
    coords = conf.GetPositions()
    return np.mean(coords, axis=0)

def main():
    print(f"🔍 TEŞHİS BAŞLIYOR...")
    
    # 1. PDB DOSYA İÇERİĞİ KONTROLÜ
    print("\n📄 PDB Dosyasının İlk 10 Satırı:")
    print("-" * 40)
    with open(PROTEIN_PDB, 'r') as f:
        for i, line in enumerate(f):
            if i >= 10: break
            print(line.strip())
    print("-" * 40)

    # 2. KOORDİNAT KONTROLÜ
    print("\n📏 Koordinat Kontrolü:")
    
    # Proteini oku
    prot_mol = Chem.MolFromPDBFile(PROTEIN_PDB, sanitize=False, removeHs=False)
    if not prot_mol:
        print("❌ HATA: RDKit PDB dosyasını okuyamadı! Format bozuk olabilir.")
        # OpenBabel bazen RDKit'in sevmediği formatlar üretir.
    else:
        prot_center = get_centroid(prot_mol)
        print(f"✅ Protein Merkezi: {prot_center}")

    # Ligandı oku
    lig_suppl = Chem.SDMolSupplier(LIGAND_SDF, sanitize=False)
    lig_mol = lig_suppl[0]
    if not lig_mol:
        print("❌ HATA: Ligand okunamadı.")
        return
        
    lig_center = get_centroid(lig_mol)
    print(f"✅ Ligand Merkezi:  {lig_center}")

    # Mesafeyi ölç
    if prot_mol and lig_mol:
        dist = np.linalg.norm(prot_center - lig_center)
        print(f"\n📐 Protein-Ligand Arasındaki Mesafe: {dist:.2f} Angstrom")
        
        if dist > 20.0:
            print("🚨 KRİTİK SORUN: Protein ve Ligand birbirinden ÇOK UZAK!")
            print("   Çözüm: Ligand üretimi sırasında koordinat taşıma (translation) işlemi başarısız olmuş.")
        else:
            print("✅ Mesafe makul görünüyor (Hizalanmış).")
            print("🚨 O zaman sorun büyük ihtimalle PDB dosyasındaki atom/residue isimlendirmesinde.")

if __name__ == "__main__":
    main()

In [ ]:
import json
import os
import sys
import traceback
from rdkit import Chem

# 1. Chemutils Patchli Versiyonu Yukle
sys.path.append("/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/Dataset_preparation")
try:
    from chemutils_dude import ComplexPocket
    print("✅ chemutils_dude (PATCHED) yuklendi.")
except ImportError:
    print("❌ chemutils_dude BULUNAMADI! Patch scripti calistirilmamis.")
    sys.exit(1)

# Ayarlar
TEST_TARGET = "ADRB2"
JSON_FILE = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/lit_pcba_paths_prepared_pdb.json"

def main():
    print(f"\n🔍 DEBUG BASLIYOR: {TEST_TARGET} hedefini test ediyoruz...")
    
    if not os.path.exists(JSON_FILE):
        print(f"❌ JSON Dosyasi Yok: {JSON_FILE}")
        return

    with open(JSON_FILE, 'r') as f:
        data = json.load(f)
    
    if TEST_TARGET not in data:
        print(f"❌ JSON icinde {TEST_TARGET} bulunamadi!")
        return

    record = data[TEST_TARGET][0]
    protein_path = record['protein_file']
    actives_path = record['actives_file']
    
    print(f"📂 Protein Yolu: {protein_path}")
    print(f"📂 Actives Yolu: {actives_path}")

    # 1. Dosyalarin Varligini Kontrol Et
    if not os.path.exists(protein_path):
        print("❌ HATA: Protein dosyasi fiziksel olarak yok!")
    else:
        print("✅ Protein dosyasi mevcut.")

    if not os.path.exists(actives_path):
        print("❌ HATA: Actives SDF dosyasi fiziksel olarak yok! (SDF uretme adimi basarisiz olmus olabilir)")
        return
    else:
        print("✅ Actives SDF dosyasi mevcut.")

    # 2. SDF Okuma Testi
    print("\n🧪 Test 1: SDF Dosyasi Okunuyor...")
    mols = []
    try:
        suppl = Chem.SDMolSupplier(actives_path, sanitize=False)
        for i, m in enumerate(suppl):
            if i >= 1: break # Sadece ilk molekule bak
            if m:
                try:
                    Chem.SanitizeMol(m)
                    mols.append(m)
                    print(f"✅ Molekul {i} okundu: {m.GetNumAtoms()} atom")
                except:
                    print(f"⚠️ Molekul {i} Sanitize edilemedi!")
            else:
                print(f"❌ Molekul {i} RDKit tarafindan okunamadi (None dondu).")
    except Exception as e:
        print(f"❌ SDF Okuma Hatasi: {e}")
        return

    if not mols:
        print("❌ Hicbir molekul basariyla okunamadi! SDF dosyasi bozuk veya bos olabilir.")
        return
    
    # 3. ComplexPocket Olusturma Testi
    print("\n🧪 Test 2: ComplexPocket olusturuluyor...")
    mol = mols[0]
    try:
        # Patchli chemutils dosya yolu yerine Mol objesi kabul etmeli
        pocket = ComplexPocket(protein_path, mol, pK=5.0)
        print("🎉 BASARILI! ComplexPocket olustu.")
        print(f"Cep bilgisi: {pocket}")
    except Exception as e:
        print("❌ ComplexPocket HATASI:")
        print("-" * 30)
        traceback.print_exc()
        print("-" * 30)

if __name__ == "__main__":
    main()


In [ ]:
import os
import sys
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm

# AYARLAR
BASE_DIR = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA"
LIMIT_MOLECULES = 1000 

# --- HEDEF PROTEİN KODU ---
# JSON dosyasında "3p0g_protein.pdb" kullandığımız için buraya o kodu yazıyoruz.
TARGET_CODE = "3p0g" 

def get_centroid(mol):
    try:
        conf = mol.GetConformer()
        pos = conf.GetPositions()
        return np.mean(pos, axis=0)
    except:
        return None

def get_reference_center_by_name(target_path, code):
    """
    Rastgele değil, özellikle (code)_ligand.mol2 dosyasını arar.
    """
    expected_name = f"{code}_ligand.mol2"
    full_path = os.path.join(target_path, expected_name)
    
    print(f"🎯 Hedef Referans Ligand Aranıyor: {expected_name}")
    
    if not os.path.exists(full_path):
        print(f"❌ Bulunamadı: {full_path}")
        return None

    # RDKit ile okumayı dene
    mol = Chem.MolFromMol2File(full_path, sanitize=False)
    if mol:
        center = get_centroid(mol)
        if center is not None:
            return center

    # Okunamazsa OpenBabel devreye girsin
    print("⚠️ RDKit okuyamadı, OpenBabel deneniyor...")
    temp_pdb = full_path.replace(".mol2", "_temp_ref.pdb")
    os.system(f"obabel {full_path} -O {temp_pdb} 2>/dev/null")
    
    if os.path.exists(temp_pdb):
        mol = Chem.MolFromPDBFile(temp_pdb, sanitize=False)
        center = get_centroid(mol)
        os.remove(temp_pdb)
        return center

    return None

def process_smiles_to_sdf(smiles_path, target_center, output_sdf_path):
    print(f"⚙️  İşleniyor: {os.path.basename(smiles_path)}")
    
    with open(smiles_path, 'r') as f:
        lines = f.readlines()

    writer = Chem.SDWriter(output_sdf_path)
    count = 0
    process_lines = lines[:LIMIT_MOLECULES] if LIMIT_MOLECULES else lines

    for line in tqdm(process_lines, desc="   Hizalanıyor"):
        parts = line.split()
        if not parts: continue
        smiles = parts[0]
        name = parts[1] if len(parts) > 1 else f"mol_{count}"

        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None: continue
            mol = Chem.AddHs(mol)
            mol.SetProp("_Name", name)

            res = AllChem.EmbedMolecule(mol, AllChem.ETKDG())
            if res != 0: continue 

            # TAŞIMA (Translation)
            mol_center = get_centroid(mol)
            if mol_center is None: continue
            
            translation = target_center - mol_center
            
            conf = mol.GetConformer()
            for i in range(mol.GetNumAtoms()):
                p = conf.GetAtomPosition(i)
                conf.SetAtomPosition(i, p + translation)

            writer.write(mol)
            count += 1
        except:
            continue

    writer.close()
    print(f"✅ {count} molekül {TARGET_CODE} merkezine hizalandı.\n")

def main():
    # Eski dosyaları temizle
    target_path = os.path.join(BASE_DIR, "ADRB2")
    if os.path.exists(os.path.join(target_path, "actives.sdf")):
        os.remove(os.path.join(target_path, "actives.sdf"))
    if os.path.exists(os.path.join(target_path, "decoys.sdf")):
        os.remove(os.path.join(target_path, "decoys.sdf"))

    # 1. Doğru Merkezi Bul (3p0g)
    target_center = get_reference_center_by_name(target_path, TARGET_CODE)
    
    if target_center is None:
        print("❌ Kritik Hata: Referans merkez bulunamadı!")
        return
    
    print(f"📍 Merkez Koordinat: {target_center}")

    # 2. Dosyaları İşle
    actives_smi = os.path.join(target_path, "actives.smi")
    inactives_smi = os.path.join(target_path, "inactives.smi")
    
    actives_sdf = os.path.join(target_path, "actives.sdf")
    decoys_sdf = os.path.join(target_path, "decoys.sdf")

    if os.path.exists(actives_smi):
        process_smiles_to_sdf(actives_smi, target_center, actives_sdf)
        
    if os.path.exists(inactives_smi):
        process_smiles_to_sdf(inactives_smi, target_center, decoys_sdf)

if __name__ == "__main__":
    main()

In [ ]:
import pickle
import os
import sys
import numpy as np
import random
# 1. Chemutils Patchli Versiyonu Yukle
sys.path.append("/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/Dataset_preparation")

from chemutils import ComplexPocket
# Senin train.pkl dosyanın yolu (Bunu kendi yoluna göre düzenle)
TRAIN_PKL_PATH = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/index/train.pkl"

def inspect_data():
    print(f"📂 Veri listesi okunuyor: {TRAIN_PKL_PATH}")
    
    if not os.path.exists(TRAIN_PKL_PATH):
        print("❌ HATA: train.pkl dosyası bulunamadı!")
        return

    with open(TRAIN_PKL_PATH, 'rb') as f:
        data_list = pickle.load(f)

    print(f"✅ Listede {len(data_list)} adet kayıt var.")
    
    # Rastgele bir dosya seçelim (Binder olsun, Decoy olmasın)
    # data_list formatı: [dosya_yolu, pK_degeri, is_decoy]
    sample = None
    for item in data_list:
        if not item[2]: # Decoy olmayan bir tane bulalım (Binder)
            sample = item
            break
            
    if sample is None:
        sample = data_list[0] # Hiç binder yoksa ilkini al

    file_path = sample[0]
    print(f"\n🔬 İNCELENEN DOSYA: {file_path}")
    
    if not os.path.exists(file_path):
        print("❌ HATA: Bu pkl dosyası diskte yok!")
        return

    with open(file_path, 'rb') as f:
        content = pickle.load(f)

    # Content bazen liste olabilir (Decoy dosyasıysa), bazen tek obje (Binder)
    obj = content
    if isinstance(content, list):
        obj = content[0]
        print("ℹ️ Bu bir Liste (Muhtemelen Decoy dosyası). İlk elemanına bakıyoruz.")

    # --- KRİTİK KONTROL ---
    print("\n🧐 İÇERİK ANALİZİ:")
    
    # 1. ESM Özelliği Var mı?
    if hasattr(obj, 'res_features'):
        print("⚠️ BULGULAR: Nesnede 'res_features' (ESM Embedding) bulundu!")
        feat = obj.res_features
        if isinstance(feat, (np.ndarray, list)) or hasattr(feat, 'shape'):
            try:
                shape = feat.shape if hasattr(feat, 'shape') else len(feat)
                print(f"🚨 VEKTÖR BOYUTU: {shape}")
                
                if shape[-1] == 1280 or shape[-1] == 512:
                    print("\n🔴 SONUÇ: BU VERİ 'ESM-2' VERİSİDİR!")
                    print("   Orijinal model (BLOSUM62) bu veriyi kullanamaz.")
                    print("   Çözüm: Veri setini 'chemutils.py' (orijinal) kullanarak yeniden oluşturmalısın.")
            except:
                pass
    else:
        print("🟢 BULGULAR: Nesnede 'res_features' YOK.")
        print("   Bu veri Orijinal (Non-ESM) formatta görünüyor.")
        
        # BLOSUM Kontrolü
        if hasattr(obj, 'res') or hasattr(obj, 'sequence'):
            print("   Nesne 'sequence' veya 'res' bilgisi taşıyor (BLOSUM için uygun).")
            print("\n✅ SONUÇ: BU VERİ 'ORİJİNAL' (Non-ESM) VERİSİDİR.")
            print("   Eğer hala OOM alıyorsan, sorun veride değil, 'train.py' argümanlarındadır (-s 300 verilmemiştir).")

if __name__ == "__main__":
    inspect_data()

In [ ]:
import pandas as pd
df = pd.read_csv("/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/decoy_candidates_find_1000_suitable.csv")
# Örnek: '1a07' için decoy var mı bak
print(len(df[df['binder_pdb_id'] == '1g1e']))

In [ ]:
import pandas as pd
import os
import subprocess

# Ayarlar (Senin sistemindeki yollar)
pdb_id = '1d4p'
inference_script = '/arf/scratch/hnaydin/banana/inference.py'
pocket_path = f'/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/banana_pocket_pdbs_12A/{pdb_id}_pocket.pdb'
temp_smi = f'temp_smi_files/{pdb_id}_candidates.smi'
out_txt = 'test_debug_scores.txt'

print(f'>>> TEST BAŞLADI: {pdb_id}')

# 1. SMILES Dosyası var mı kontrolü
if not os.path.exists(temp_smi):
    print(f'HATA: {temp_smi} dosyası bulunamadı.')
    print('Lütfen "temp_smi_files" klasörünün içeriğini "ls temp_smi_files" ile kontrol et.')
else:
    # 2. BANANA'yı manuel tetikle
    cmd = ['python', inference_script, temp_smi, pocket_path, '--out_file', out_txt, '--no_gpu']
    
    # f-string hatasından kaçınmak için join işlemini dışarıda yapıyoruz
    cmd_str = " ".join(cmd)
    print(f'>>> KOMUT ÇALIŞTIRILIYOR: {cmd_str}')
    
    # İşlemi çalıştır ve hata/çıktıları yakala
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    print('\n--- STDOUT (ÇIKTI) ---')
    print(result.stdout if result.stdout else "Çıktı yok.")
    
    print('\n--- STDERR (HATA MESAJI) ---')
    if result.stderr:
        print(result.stderr)
    else:
        print("Hata mesajı yok.")
    
    # 3. Sonuç Kontrolü
    if os.path.exists(out_txt):
        print(f'\n✅ BAŞARILI: {out_txt} dosyası başarıyla oluştu!')
    else:
        print('\n❌ HATA: Komut çalıştı bitti ama fiziksel bir dosya oluşmadı.')

In [ ]:
import os
import subprocess

# --- TAM (ABSOLUTE) YOLLAR ---
pdb_id = '1d4p'
base_path = '/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model'
inference_script = '/arf/scratch/hnaydin/banana/inference.py'

temp_smi = os.path.join(base_path, 'temp_smi_files', f'{pdb_id}_candidates.smi')
pocket_pdb = os.path.join(base_path, 'datasets/pdbbind2020/banana_pocket_pdbs_12A', f'{pdb_id}_pocket.pdb')
out_txt = os.path.join(base_path, 'test_debug_scores.txt')

print(f"🔍 Kontrol Edilen Dosya: {temp_smi}")

if not os.path.exists(temp_smi):
    print(f"❌ HATA: Dosya hala bulunamadı! Lütfen terminalde 'readlink -f temp_smi_files' komutuyla tam yolu doğrula.")
else:
    print("✅ SMILES dosyası bulundu. BANANA (Inference) başlatılıyor...")
    
    # BANANA Komutu
    cmd = ['python', inference_script, temp_smi, pocket_pdb, '--out_file', out_txt, '--no_gpu']
    
    print(f">>> ÇALIŞTIRILAN KOMUT: {' '.join(cmd)}")
    
    # Çalıştır ve hataları yakala
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    print('\n' + '='*30)
    print('--- STDOUT (BANANA Çıktısı) ---')
    print(result.stdout if result.stdout else "(Çıktı yok)")
    
    print('\n' + '='*30)
    print('--- STDERR (HATA MESAJI) ---')
    if result.stderr:
        print(result.stderr)
    else:
        print("Harika! Hiçbir hata mesajı gelmedi.")
    
    if os.path.exists(out_txt):
        print(f"\n✨ BİNGO! '{out_txt}' dosyası oluştu. Sorun yokmuş, sadece sıra gelmemiş.")
    else:
        print("\n⚠️ DİKKAT: Komut bitti ama dosya oluşmadı. Yukarıdaki hata mesajını incele.")

In [ ]:
import terrace

In [ ]:
import pandas as pd
import os

# 1. Klasördeki mevcut sonuçları al
scores_dir = '/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/banana_output_scores_filtered/'
existing_ids = {f.split('_')[0] for f in os.listdir(scores_dir) if f.endswith('.csv')}

# 2. Decoy CSV'sini yükle
df = pd.read_csv('/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/decoy_candidates_find_1000_suitable.csv')
all_decoys_ids = set(df['binder_pdb_id'].unique())

# 3. Klasörde olmayıp CSV'de olan var mı bak
missing_ids = all_decoys_ids - existing_ids

if not missing_ids:
    print('Kritik Durum: Skorlanacak yeni PDB\'lerin hiçbirine ait decoy CSV\'de bulunamadı!')
else:
    test_id = list(missing_ids)[0]
    count = len(df[df['binder_pdb_id'] == test_id])
    print(f'Test Edilen PDB: {test_id}')
    print(f'Bulunan Decoy Sayısı: {count}')

In [ ]:
import sys
import os
import numpy as np
from rdkit import Chem

# --- AYARLAR ---
CHEMUTILS_PATH = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/Dataset_preparation"
sys.path.append(CHEMUTILS_PATH)

# ALDH1 yollarini senin verdigin ls ciktisina gore el ile yaziyorum (Garanti olsun)
PROTEIN_PATH = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/ALDH1/5l2n_protein.pdb"
ACTIVES_SDF = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/ALDH1/actives.sdf"

print(f"📂 Protein: {PROTEIN_PATH}")
print(f"📂 Ligandlar: {ACTIVES_SDF}")

# Chemutils yukle
try:
    from chemutils_dude import ComplexPocket
    print("✅ Chemutils yüklendi.")
except ImportError:
    print("❌ Chemutils yüklenemedi! Yolu kontrol et.")
    sys.exit(1)

def test_single_pocket():
    # 1. SDF Oku
    if not os.path.exists(ACTIVES_SDF):
        print("❌ SDF bulunamadi!")
        return

    suppl = Chem.SDMolSupplier(ACTIVES_SDF, sanitize=False)
    mols = [m for m in suppl if m is not None]
    print(f"🧪 SDF içinden {len(mols)} molekül okundu.")
    
    if len(mols) == 0:
        print("❌ Molekül sayısı 0! Sorun SDF dosyasının içinde.")
        return

    # 2. İLK MOLEKÜLÜ AL VE ZORLA
    mol = mols[0]
    print("\n⚡ ComplexPocket Testi Başlıyor (İlk molekül için)...")
    
    try:
        # Hata yakalama bloğu OLMADAN çalıştırıyoruz
        pocket = ComplexPocket(PROTEIN_PATH, mol, pK=5.0)
        print("✅ BAŞARILI! Cep oluşturuldu.")
        print(f"   -> Atom sayısı: {len(pocket.ligand_atoms)}")
        print(f"   -> Residue sayısı: {len(pocket.pocket_residues)}")
    except Exception as e:
        print("\n❌ ❌ ❌ HATA YAKALANDI! ❌ ❌ ❌")
        print("ComplexPocket şu hatayı verdi:")
        print("-" * 30)
        print(e)
        print("-" * 30)
        import traceback
        traceback.print_exc()
        print("-" * 30)
        print("💡 İPUCU: Hata mesajını kopyalayıp bana at!")

if __name__ == "__main__":
    test_single_pocket()

In [ ]:
import os
import numpy as np
from rdkit import Chem

# --- YOLLAR ---
PROTEIN_PDB = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/ALDH1/5l2n_protein.pdb"
ACTIVES_SDF = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/ALDH1/actives.sdf"
REF_LIGAND = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/ALDH1/5l2n_ligand.mol2"

def get_pdb_center(pdb_path):
    coords = []
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                try:
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    coords.append([x, y, z])
                except:
                    pass
    if not coords: return None
    return np.mean(coords, axis=0)

def get_sdf_center(sdf_path):
    suppl = Chem.SDMolSupplier(sdf_path, sanitize=False)
    for mol in suppl:
        if mol:
            conf = mol.GetConformer()
            pos = conf.GetPositions()
            return np.mean(pos, axis=0)
    return None

def main():
    print("📏 MESAFE KONTROLU (ALDH1)")
    
    # 1. Referans Dosya Var mı?
    if os.path.exists(REF_LIGAND):
        print(f"✅ Referans Ligand Dosyası MEVCUT: {os.path.basename(REF_LIGAND)}")
    else:
        print(f"❌ Referans Ligand Dosyası YOK! ({os.path.basename(REF_LIGAND)})")
        print("   -> Hizalama scripti muhtemelen başka bir dosyayı baz aldı!")

    # 2. Merkezleri Hesapla
    prot_center = get_pdb_center(PROTEIN_PDB)
    lig_center = get_sdf_center(ACTIVES_SDF)

    if prot_center is None:
        print("❌ Protein okunurken hata oldu.")
        return
    if lig_center is None:
        print("❌ SDF okunurken hata oldu.")
        return

    print(f"\n📍 Protein Merkezi: {prot_center}")
    print(f"📍 Ligand Merkezi:  {lig_center}")

    # 3. Mesafeyi Ölç
    dist = np.linalg.norm(prot_center - lig_center)
    print(f"\n📏 ARADAKİ MESAFE: {dist:.2f} Angstrom")

    if dist > 10:
        print("\n🚨 KRİTİK SORUN: Ligandlar proteinden çok uzak!")
        print("   Bu yüzden 'concatenate' hatası alıyorsun.")
    else:
        print("\n✅ Mesafe makul görünüyor.")

if __name__ == "__main__":
    main()

In [ ]:
import os
import sys

# --- AYARLAR ---
# Sorunlu oldugunu bildigimiz dosya (Test icin)
TARGET_PDB = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/ALDH1/5l2n_protein.pdb"
BACKUP_PDB = TARGET_PDB + ".bak"

def fix_pdb_file(pdb_path):
    print(f"🔧 PDB Onarılıyor: {os.path.basename(pdb_path)}")
    
    if not os.path.exists(pdb_path):
        print("❌ Dosya bulunamadı.")
        return False

    with open(pdb_path, 'r') as f:
        lines = f.readlines()

    fixed_lines = []
    atom_count = 0
    fixed_count = 0

    for line in lines:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            atom_count += 1
            # PDB Standardı (Sütunlar):
            # 0-6: Kayıt Adı (ATOM  )
            # 21: Zincir Belirteci (Chain ID) -> Genelde burasi boştur
            
            # Eğer satır çok kısaysa atla
            if len(line) < 54: 
                fixed_lines.append(line)
                continue

            # Zincir ID (21. karakter, Python index 21) boşsa 'A' yap
            # Python stringleri değiştirilemez, listeye çevirip değiştiriyoruz
            line_list = list(line)
            
            # Chain ID kontrolü (Index 21)
            if line_list[21] == ' ':
                line_list[21] = 'A'
                fixed_count += 1
            
            fixed_lines.append("".join(line_list))
        else:
            fixed_lines.append(line)

    # Yedek al (Eğer daha önce almadıysak)
    if not os.path.exists(BACKUP_PDB):
        os.rename(pdb_path, BACKUP_PDB)
        print(f"   💾 Yedek alındı: {os.path.basename(BACKUP_PDB)}")
    
    # Dosyayı üzerine yaz
    with open(pdb_path, 'w') as f:
        f.writelines(fixed_lines)

    print(f"✅ İşlem tamam.")
    print(f"   -> Toplam Atom: {atom_count}")
    print(f"   -> Onarılan Zincir ID: {fixed_count}")
    return True

if __name__ == "__main__":
    fix_pdb_file(TARGET_PDB)

In [ ]:
import os
import sys

# Hata veren ALDH1 dosyasi
TARGET_PDB = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/ALDH1/5l2n_protein.pdb"

def fix_chain_id(pdb_path):
    print(f"🔧 PDB Onarılıyor: {pdb_path}")
    
    if not os.path.exists(pdb_path):
        print("❌ Dosya bulunamadı!")
        return

    # 1. Dosyayı Oku
    with open(pdb_path, 'r') as f:
        lines = f.readlines()

    fixed_lines = []
    count = 0

    for line in lines:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            # PDB formatında 21. karakter (index 21) Chain ID'dir.
            # Eğer satır kısaysa veya o hane boşsa müdahale et.
            if len(line) > 21:
                if line[21] == ' ':
                    # Stringi listeye çevir (değiştirmek için)
                    l = list(line)
                    l[21] = 'A' # Boşluğu 'A' yap
                    line = "".join(l)
                    count += 1
            
        fixed_lines.append(line)

    # 2. Üzerine Yaz (Yedek alarak)
    os.rename(pdb_path, pdb_path + ".bak") # Yedek
    
    with open(pdb_path, 'w') as f:
        f.writelines(fixed_lines)
        
    print(f"✅ Tamamlandı! {count} atoma 'A' zincir etiketi eklendi.")

if __name__ == "__main__":
    fix_chain_id(TARGET_PDB)

In [ ]:
import os
import sys

# Hedef dosya
TARGET_PDB = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/ALDH1/5l2n_protein.pdb"

def normalize_pdb(pdb_path):
    print(f"🔧 PDB Standardizasyonu: {pdb_path}")
    
    if not os.path.exists(pdb_path):
        print("❌ Dosya bulunamadı!")
        return

    with open(pdb_path, 'r') as f:
        lines = f.readlines()

    fixed_lines = []
    changes = 0

    for line in lines:
        # Sadece atom satırlarıyla ilgileniyoruz
        if line.startswith("ATOM") or line.startswith("HETATM"):
            
            # 1. HETATM -> ATOM Dönüşümü
            # (HETATM genelde 6 harf, ATOM 4 harf + 2 boşluk)
            if line.startswith("HETATM"):
                line = "ATOM  " + line[6:]
                changes += 1

            # 2. Zincir ID (Chain A) Garantileme
            if len(line) > 21:
                if line[21] == ' ':
                    l = list(line)
                    l[21] = 'A'
                    line = "".join(l)
                    changes += 1
            
            fixed_lines.append(line)
        else:
            fixed_lines.append(line)

    # Dosyayı güncelle
    with open(pdb_path, 'w') as f:
        f.writelines(fixed_lines)
        
    print(f"✅ İşlem bitti. Toplam {changes} düzeltme yapıldı.")

if __name__ == "__main__":
    normalize_pdb(TARGET_PDB)

In [ ]:
import json
import os
import sys
from tqdm import tqdm

# --- AYARLAR ---
JSON_FILE = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/lit_pcba_paths_prepared_pdb.json"

def standardize_pdb(pdb_path):
    if not os.path.exists(pdb_path): return False, 0
    
    with open(pdb_path, 'r') as f:
        lines = f.readlines()
        
    fixed_lines = []
    changes = 0
    
    for line in lines:
        # Sadece atom satirlari ile ilgilen
        if line.startswith("ATOM") or line.startswith("HETATM"):
            
            # 1. HETATM -> ATOM
            if line.startswith("HETATM"):
                line = "ATOM  " + line[6:]
                changes += 1
                
            # 2. Zincir ID (Chain A) Garantile
            if len(line) > 21 and line[21] == ' ':
                l = list(line)
                l[21] = 'A'
                line = "".join(l)
                changes += 1
                
        fixed_lines.append(line)
        
    # Eger degisiklik yapildiysa kaydet
    if changes > 0:
        with open(pdb_path, 'w') as f:
            f.writelines(fixed_lines)
            
    return True, changes

def main():
    if not os.path.exists(JSON_FILE):
        print("❌ JSON bulunamadi!")
        return

    with open(JSON_FILE, 'r') as f:
        data = json.load(f)
        
    print(f"🔧 Tüm hedefler için PDB Standardizasyonu başlıyor...")
    
    total_files = 0
    total_fixes = 0
    
    for target, records in tqdm(data.items()):
        if not records: continue
        pdb_path = records[0]['protein_file']
        
        success, change_count = standardize_pdb(pdb_path)
        if success and change_count > 0:
            total_files += 1
            total_fixes += change_count
            
    print(f"\n✅ İŞLEM TAMAMLANDI!")
    print(f"   -> Onarılan Dosya: {total_files}")
    print(f"   -> Toplam Düzeltme: {total_fixes}")
    print("🚀 Artık PKL üretimini başlatabilirsin!")

if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import pickle
import random
import glob

# AYARLAR
BASE_DIR = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020"
NEW_DATA_DIR = os.path.join(BASE_DIR, "planet_esm2_pkl")
JSON_MAP = os.path.join(BASE_DIR, "PDBbind_structure_map_pkd_prepared.json")

# Çıktı Dosyaları
OUT_TRAIN = "train_esm2.pkl"
OUT_VALID = "valid_esm2.pkl"

def create_lists():
    with open(JSON_MAP, 'r') as f:
        data_map = json.load(f)

    train_data = []
    valid_data = []

    print("Listeler oluşturuluyor...")
    
    for pdb_id, info in data_map.items():
        split = info.get('split', 'train') # train, valid, test
        pK = float(info.get('pK', 0.0))
        
        # PDB dosyasının orijinal yolundan "yıl klasörünü" buluyoruz
        # Orijinal: .../2011-2019/1lox/...
        orig_pdb_path = info.get('protein_pdb')
        if not orig_pdb_path: continue
        
        # Yıl klasörünü path'ten ayıkla (path yapına göre değişebilir, genelde datasets/pdbbind2020'den sonra gelir)
        # Basit yöntem: JSON'daki path'in parçalarını kullan
        parts = orig_pdb_path.split('/')
        year_folder = parts[-3] # Genelde sondan 3. klasör yıldır (2011-2019)
        
        # Yeni Dosya Yollarını Oluştur
        pdb_dir = os.path.join(NEW_DATA_DIR, year_folder, pdb_id)
        
        # Dosya isimleri (ESM-2 sonekine dikkat!)
        binder_file = os.path.join(pdb_dir, f"{pdb_id}_binder_esm512d.pkl")
        decoy_file = os.path.join(pdb_dir, f"{pdb_id}_decoys_esm512d.pkl")
        
        if not os.path.exists(binder_file):
            continue # Dosya yoksa listeye ekleme
            
        # Listeye Ekle: [Dosya_Yolu, Label (pK veya 0)]
        entry_list = []
        
        # 1. Binder Ekle
        entry_list.append([binder_file, pK])
        
        # 2. Decoys Ekle (Eğer train veya valid ise)
        if split in ['train', 'valid'] and os.path.exists(decoy_file):
            # Decoy dosyası listeyse tek path veriyoruz, datautils bunu açmalı.
            # Ancak senin datautils yapın muhtemelen "her decoy için bir satır" istiyor olabilir.
            # Eğer datautils'in "tek dosya çok decoy" desteklemiyorsa burası karışabilir.
            # *VARSAYIM:* PLANET_datautils, decoy pkl dosyasının içinden rastgele seçim yapıyordur.
            # Biz buraya decoy dosyasının yolunu veriyoruz, label 0.0
            entry_list.append([decoy_file, 0.0])

        if split == 'train':
            train_data.extend(entry_list)
        elif split == 'valid':
            valid_data.extend(entry_list)

    # Kaydet
    with open(OUT_TRAIN, 'wb') as f:
        pickle.dump(train_data, f)
        
    with open(OUT_VALID, 'wb') as f:
        pickle.dump(valid_data, f)

    print(f"Tamamlandı! Train: {len(train_data)}, Valid: {len(valid_data)}")

if __name__ == "__main__":
    create_lists()

In [3]:
import sys
sys.path.append("/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/Dataset_preparation")
try:
    from chemutils_esm2 import ComplexPocket
    print("✅ chemutils_dude (PATCHED) yuklendi.")
except ImportError:
    print("❌ chemutils_dude BULUNAMADI! Patch scripti calistirilmamis.")
    sys.exit(1)
    
import pickle
import torch

# ESM'li bir dosya yolu seçelim
esm_file = '/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/planet_esm2_pkl/4tjy_binder_esm512d.pkl'

with open(esm_file, 'rb') as f:
    data = pickle.load(f)

print(f"--- ESM Dosya İncelemesi ---")
print(f"Veri Tipi: {type(data)}")

if isinstance(data, list):
    print(f"Liste Uzunluğu: {len(data)}")
    print(f"İlk Eleman Tipi: {type(data[0])}")
    if hasattr(data[0], 'shape'): print(f"Boyut (Shape): {data[0].shape}")
elif isinstance(data, dict):
    print(f"Sözlük Anahtarları (Keys): {data.keys()}")

✅ chemutils_dude (PATCHED) yuklendi.
--- ESM Dosya İncelemesi ---
Veri Tipi: <class 'chemutils.ComplexPocket'>


In [4]:
import pickle
import sys

# Yolları ekleyelim
sys.path.append("/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/Dataset_preparation")

# İki farklı sınıfı farklı isimlerle import edelim
try:
    from chemutils import ComplexPocket as ComplexPocket_NoESM
    from chemutils_esm2 import ComplexPocket as ComplexPocket_ESM
    print("✅ Her iki sınıf da başarıyla yüklendi.\n")
except ImportError as e:
    print(f"❌ Import hatası: {e}")

# Dosya yolları
esm_byte_path = '/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/planet_esm2_pkl/4tjy_decoys_byte_esm512d.pkl'
no_esm_byte_path = '/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all_pickles/4tjy_decoys_byte.pkl'

def analiz_et(byte_path, pkl_path, isim):
    try:
        with open(byte_path, 'rb') as f:
            offsets = pickle.load(f)
        
        print(f"--- {isim} Analizi ---")
        print(f"İndekslenen Nesne Sayısı: {len(offsets)}")
        
        if len(offsets) > 1:
            # İki offset arasındaki fark, kabaca bir nesnenin diskteki boyutudur
            ortalama_boyut = (offsets[1] - offsets[0]) / 1024 # KB cinsinden
            print(f"Tek bir nesnenin yaklaşık boyutu: {ortalama_boyut:.2f} KB")
        
        # Gerçek pkl dosyasının toplam boyutu
        import os
        total_size = os.path.getsize(pkl_path.replace("_byte", "")) / (1024 * 1024) # MB
        print(f"Toplam PKL boyutu: {total_size:.2f} MB")
        print("-" * 30)
        
    except Exception as e:
        print(f"{isim} hatası: {e}")

# 4tjy örneği üzerinden karşılaştıralım
analiz_et(esm_byte_path, esm_byte_path, "ESM'li (ComplexPocket_ESM)")
analiz_et(no_esm_byte_path, no_esm_byte_path, "ESM'siz (ComplexPocket_NoESM)")

✅ Her iki sınıf da başarıyla yüklendi.

--- ESM'li (ComplexPocket_ESM) Analizi ---
İndekslenen Nesne Sayısı: 386
Tek bir nesnenin yaklaşık boyutu: 66.38 KB
Toplam PKL boyutu: 24.99 MB
------------------------------
--- ESM'siz (ComplexPocket_NoESM) Analizi ---
İndekslenen Nesne Sayısı: 386
ESM'siz (ComplexPocket_NoESM) hatası: unsupported operand type(s) for -: 'bytes' and 'bytes'


In [5]:
import pickle

no_esm_byte = '/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all_pickles/4tjy_decoys_byte.pkl'

with open(no_esm_byte, 'rb') as f:
    data = pickle.load(f)

print(f"Nesne Tipi: {type(data)}")
if isinstance(data, list):
    print(f"Listenin Uzunluğu: {len(data)}")
    print(f"Listenin ilk elemanının tipi: {type(data[0])}")
    # Eğer ilk eleman bir sayı değilse, içindeki veriyi görelim
    if not isinstance(data[0], int):
        print(f"İlk eleman içeriği (ilk 100 karakter): {str(data[0])[:100]}")

Nesne Tipi: <class 'list'>
Listenin Uzunluğu: 386
Listenin ilk elemanının tipi: <class 'bytes'>
İlk eleman içeriği (ilk 100 karakter): b'\x80\x05\x95c\t\x01\x00\x00\x00\x00\x00\x8c\tchemutils\x94\x8c\rComplexPocket\x94\x93\x94)\x81\x94


In [1]:
import os
from rdkit import Chem
from rdkit.Chem import AllChem

# Klasör yolu
base_path = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA"

def convert_files():
    targets = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d)) and d not in ['benchmark_results', 'lit_pcba_pkl_data']]
    
    for target in targets:
        target_dir = os.path.join(base_path, target)
        print(f"🔄 İşleniyor: {target}")
        
        for file in os.listdir(target_dir):
            file_path = os.path.join(target_dir, file)
            
            # 1. Protein Dönüşümü: mol2 -> pdb
            if file.endswith("_protein.mol2"):
                out_pdb = file_path.replace(".mol2", ".pdb")
                if not os.path.exists(out_pdb):
                    prot = Chem.MolFromMol2File(file_path, sanitize=False, removeHs=False)
                    if prot:
                        Chem.MolToPDBFile(prot, out_pdb)
                        print(f"  ✅ Protein kaydedildi: {os.path.basename(out_pdb)}")
            
            # 2. Referans Ligand Dönüşümü: mol2 -> sdf
            elif file.endswith("_ligand.mol2"):
                out_sdf = file_path.replace(".mol2", ".sdf")
                if not os.path.exists(out_sdf):
                    lig = Chem.MolFromMol2File(file_path, sanitize=True, removeHs=False)
                    if lig:
                        writer = Chem.SDWriter(out_sdf)
                        writer.write(lig)
                        writer.close()
                        print(f"  ✅ Ligand kaydedildi: {os.path.basename(out_sdf)}")

if __name__ == "__main__":
    convert_files()
    print("\n✨ Tüm dönüşümler tamamlandı!")

🔄 İşleniyor: OPRK1
  ✅ Ligand kaydedildi: 6b73_ligand.sdf
🔄 İşleniyor: FEN1
  ✅ Ligand kaydedildi: 5fv7_ligand.sdf
🔄 İşleniyor: ALDH1


[18:35:30] Can't kekulize mol.  Unkekulized atoms: 4 8 23 24 26
[18:35:31] sanitize [18:35:31] ****: [18:35:31] Explicit valence for atom # 22 C, 5, is greater than permitted
[18:35:31] sanitize [18:35:31] ****: [18:35:31] Explicit valence for atom # 18 C, 5, is greater than permitted
[18:35:31] sanitize [18:35:31] ****: [18:35:31] ****: Error - C.Cat with bad number of N neighbors.
[18:35:31] Can't kekulize mol.  Unkekulized atoms: 3 7 8 10 14 19 20 23 24
[18:35:31] sanitize [18:35:31] ****: [18:35:31] Can't kekulize mol.  Unkekulized atoms: 46 47 48 49 50 51 52 65 66
[18:35:31] sanitize [18:35:31] ****: [18:35:31] ****: Error - C.Cat with bad number of N neighbors.
[18:35:31] ****: Error - C.Cat with bad number of N neighbors.
[18:35:31] Can't kekulize mol.  Unkekulized atoms: 6 7 8 10 11 12 16 18 19
[18:35:31] sanitize [18:35:31] ****: [18:35:31] Can't kekulize mol.  Unkekulized atoms: 2 3 26 27 32 36 37 39 43
[18:35:31] sanitize [18:35:31] ****: [18:35:31] ****: Error - C.Cat with 

  ✅ Ligand kaydedildi: 5l2n_ligand.sdf
  ✅ Ligand kaydedildi: 5l2m_ligand.sdf
  ✅ Ligand kaydedildi: 5ac2_ligand.sdf
  ✅ Ligand kaydedildi: 5l2o_ligand.sdf
🔄 İşleniyor: ESR1_ant
  ✅ Ligand kaydedildi: 6b0f_ligand.sdf
  ✅ Ligand kaydedildi: 5fqv_ligand.sdf
  ✅ Ligand kaydedildi: 5ufx_ligand.sdf
  ✅ Ligand kaydedildi: 2iog_ligand.sdf
  ✅ Ligand kaydedildi: 5t92_ligand.sdf
  ✅ Ligand kaydedildi: 2iok_ligand.sdf
  ✅ Ligand kaydedildi: 2ouz_ligand.sdf
  ✅ Ligand kaydedildi: 1xp1_ligand.sdf
  ✅ Ligand kaydedildi: 2pog_ligand.sdf
  ✅ Ligand kaydedildi: 1xqc_ligand.sdf
  ✅ Ligand kaydedildi: 2ayr_ligand.sdf
  ✅ Ligand kaydedildi: 2r6w_ligand.sdf
  ✅ Ligand kaydedildi: 3dt3_ligand.sdf
🔄 İşleniyor: MAPK1
  ✅ Ligand kaydedildi: 4qte_ligand.sdf
  ✅ Ligand kaydedildi: 6g9h_ligand.sdf
  ✅ Ligand kaydedildi: 4xj0_ligand.sdf
  ✅ Ligand kaydedildi: 3sa0_ligand.sdf
  ✅ Ligand kaydedildi: 4qp9_ligand.sdf
  ✅ Ligand kaydedildi: 2ojg_ligand.sdf
  ✅ Ligand kaydedildi: 4zzn_ligand.sdf
  ✅ Ligand kaydedildi: 

[18:35:31] sanitize [18:35:31] ****: [18:35:31] Explicit valence for atom # 1 P, 7, is greater than permitted
[18:35:31] sanitize [18:35:31] ****: [18:35:31] Can't kekulize mol.  Unkekulized atoms: 6 7 14 15 18 25 26 27 28
[18:35:31] sanitize [18:35:31] ****: [18:35:31] Explicit valence for atom # 25 C, 5, is greater than permitted
[18:35:31] sanitize [18:35:31] ****: [18:35:31] ****: warning - charged amidine and isFixed atom.
[18:35:31] Explicit valence for atom # 56 N, 5, is greater than permitted
[18:35:31] sanitize [18:35:31] ****: [18:35:31] Can't kekulize mol.  Unkekulized atoms: 4 9 10 11 12 13 14 15 16 17 18 19 20
[18:35:31] sanitize [18:35:31] ****: [18:35:31] Can't kekulize mol.  Unkekulized atoms: 3 17 20
[18:35:31] sanitize [18:35:31] ****: [18:35:31] Can't kekulize mol.  Unkekulized atoms: 1 3 10 12 14
[18:35:31] sanitize [18:35:31] ****: [18:35:31] ****: Error - C.Cat with bad number of N neighbors.
[18:35:31] Explicit valence for atom # 12 C, 5, is greater than permitte

In [2]:
from rdkit import Chem
import os

prot_path = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/PKM2/4g1n_protein.pdb"
lig_path = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/PKM2/4g1n_ligand.sdf"

prot = Chem.MolFromPDBFile(prot_path)
lig = Chem.SDMolSupplier(lig_path)[0] if os.path.exists(lig_path) else None

if prot is None or lig is None:
    print(f"❌ Dosya okuma hatası: Protein={prot is not None}, Ligand={lig is not None}")
else:
    res_info_exists = any(a.GetPDBResidueInfo() is not None for a in prot.GetAtoms())
    print(f"✅ Residue Bilgisi Mevcut mu: {res_info_exists}")
    print(f"✅ Protein Atom Sayısı: {prot.GetNumAtoms()}")

✅ Residue Bilgisi Mevcut mu: True
✅ Protein Atom Sayısı: 3965


In [3]:
from rdkit import Chem
import numpy as np
from collections import defaultdict
import os

# 1. Dosya Yolları (Kendi sistemindeki yolları kontrol et)
target_path = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/PKM2"
prot_file = os.path.join(target_path, "4g1n_protein.pdb")
lig_file = os.path.join(target_path, "4g1n_ligand.sdf")

# Scriptindeki mevcut sözlük
AA_3_TO_1 = {'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'}

print("--- 🔍 PKM2_4g1n Pocket Debug Başlatıldı ---")

# 2. Dosyaları Yükle
prot = Chem.MolFromPDBFile(prot_file, sanitize=False)
suppl = Chem.SDMolSupplier(lig_file, sanitize=False)
ref_mol = suppl[0] if len(suppl) > 0 else None

if prot is None or ref_mol is None:
    print(f"❌ Dosya okuma hatası! Protein: {prot is not None}, Ligand: {ref_mol is not None}")
else:
    # 3. Merkez Hesapla
    center = np.mean(ref_mol.GetConformer().GetPositions(), axis=0)
    print(f"📍 Referans Ligand Merkezi: {center}")

    # 4. Yakındaki Atomları ve İsimlerini İncele
    conf_prot = prot.GetConformer()
    found_residues_in_12A = set()
    mapped_residues_count = 0
    
    print("\n📏 12 Angstrom içindeki atomlar taranıyor...")
    
    for atom in prot.GetAtoms():
        pos = conf_prot.GetAtomPosition(atom.GetIdx())
        dist = np.sqrt((pos.x - center[0])**2 + (pos.y - center[1])**2 + (pos.z - center[2])**2)
        
        if dist <= 12.0:
            info = atom.GetPDBResidueInfo()
            if info:
                res_name = info.GetResidueName().strip().upper()
                found_residues_in_12A.add(res_name)
                
                # Scriptindeki listede var mı?
                if res_name in AA_3_TO_1:
                    mapped_residues_count += 1

    # 5. Raporlama
    print("-" * 40)
    print(f"✅ 12Å mesafe içinde bulunan TÜM residue isimleri: {found_residues_in_12A}")
    print(f"✅ Bunlardan AA_3_TO_1 sözlüğünde olanların sayısı: {mapped_residues_count}")
    
    if len(found_residues_in_12A) == 0:
        print("⚠️ HATA: Ligandın 12Å çevresinde hiç protein atomu yok! (Hizalama sorunu)")
    elif mapped_residues_count == 0:
        print("⚠️ HATA: Kalıntılar bulundu ama isimleri sözlükle eşleşmiyor! (Örn: HIE, HID, ASH)")
    else:
        print("🎉 Pocket aslında geçerli görünüyor, sorun başka bir yerde olabilir.")

--- 🔍 PKM2_4g1n Pocket Debug Başlatıldı ---
📍 Referans Ligand Merkezi: [  1.40112321 -12.59167321  46.73878393]

📏 12 Angstrom içindeki atomlar taranıyor...
----------------------------------------
✅ 12Å mesafe içinde bulunan TÜM residue isimleri: {'UNL'}
✅ Bunlardan AA_3_TO_1 sözlüğünde olanların sayısı: 0
⚠️ HATA: Kalıntılar bulundu ama isimleri sözlükle eşleşmiyor! (Örn: HIE, HID, ASH)


In [5]:
import subprocess
import os

# Dosya yolları
target_path = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/PKM2"
mol2_file = os.path.join(target_path, "4g1n_protein.mol2")
pdb_output = os.path.join(target_path, "4g1n_babel_jupyter.pdb")

print("🧪 OpenBabel ile çevrim başlatılıyor...")

# 1. OpenBabel komutunu çalıştır
# -imol2: girdi formatı, -opdb: çıktı formatı, -xp: PDB kalıntı numaralarını koru
cmd = f"obabel -imol2 {mol2_file} -opdb -O {pdb_output}"

try:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ Çevrim başarıyla tamamlandı!")
        
        # 2. Dosyanın içini kontrol et
        print("\n📝 PDB Dosyasının ilk 15 satırı (Residue kontrolü):")
        print("-" * 50)
        with open(pdb_output, 'r') as f:
            lines = f.readlines()
            for line in lines[:15]:
                print(line.strip())
    else:
        print(f"❌ HATA: {result.stderr}")
        print("💡 Not: OpenBabel yüklü olmayabilir. 'module load openbabel' komutunu denemelisin.")

except Exception as e:
    print(f"❌ Beklenmedik bir hata oluştu: {e}")

🧪 OpenBabel ile çevrim başlatılıyor...


✅ Çevrim başarıyla tamamlandı!

📝 PDB Dosyasının ilk 15 satırı (Residue kontrolü):
--------------------------------------------------
COMPND    H_4g1n
AUTHOR    GENERATED BY OPEN BABEL 3.1.0
ATOM      1  N   GLN A   1     -13.295 -18.702  62.596  1.00  0.00           N
ATOM      2  CA  GLN A   1     -12.699 -19.272  61.348  1.00  0.00           C
ATOM      3  C   GLN A   1     -11.254 -19.724  61.566  1.00  0.00           C
ATOM      4  O   GLN A   1     -10.454 -19.026  62.212  1.00  0.00           O
ATOM      5  CB  GLN A   1     -12.773 -18.264  60.195  1.00  0.00           C
ATOM      6  CG  GLN A   1     -14.168 -18.069  59.608  1.00  0.00           C
ATOM      7  CD  GLN A   1     -14.548 -19.140  58.592  1.00  0.00           C
ATOM      8  OE1 GLN A   1     -14.281 -20.333  58.784  1.00  0.00           O
ATOM      9  NE2 GLN A   1     -15.185 -18.715  57.502  1.00  0.00           N
ATOM     10  H   GLN A   1     -13.671 -17.775  62.592  1.00  0.00           H
ATOM     11  HA  GL

In [6]:
import subprocess
import os
import glob

base_path = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA"

# Klasörleri listele
targets = [d for d in os.listdir(base_path) 
           if os.path.isdir(os.path.join(base_path, d)) 
           and d not in ['benchmark_results', 'lit_pcba_pkl_data']]

print(f"🧹 Temizlik ve 🚀 Onarım başlatılıyor (Toplam {len(targets)} hedef)...\n")

for target in targets:
    target_dir = os.path.join(base_path, target)
    
    # 1. ESKİ HATALI PDB'LERİ SİL
    old_pdbs = glob.glob(os.path.join(target_dir, "*.pdb"))
    for old_pdb in old_pdbs:
        try:
            os.remove(old_pdb)
        except:
            pass
    
    print(f"📦 {target}: Eskiler silindi, yeniler üretiliyor...")
    
    # 2. MOL2'LERİ BUL VE OPENBABEL İLE ÇEVİR
    mol2_files = [f for f in os.listdir(target_dir) if f.endswith("_protein.mol2")]
    
    for f in mol2_files:
        mol2_path = os.path.join(target_dir, f)
        pdb_path = mol2_path.replace(".mol2", ".pdb")
        
        # OpenBabel komutu (-xp: PDB kalıntı bilgilerini koru)
        cmd = f"obabel -imol2 {mol2_path} -opdb -O {pdb_path} -xp"
        
        try:
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            if result.returncode != 0:
                print(f"   ❌ Hata ({f}): {result.stderr}")
        except Exception as e:
            print(f"   ❌ Beklenmedik hata ({f}): {e}")

print("\n✨ LIT-PCBA klasörü tamamen temizlendi ve OpenBabel ile onarıldı!")

🧹 Temizlik ve 🚀 Onarım başlatılıyor (Toplam 16 hedef)...

📦 OPRK1: Eskiler silindi, yeniler üretiliyor...
📦 FEN1: Eskiler silindi, yeniler üretiliyor...
📦 ALDH1: Eskiler silindi, yeniler üretiliyor...
📦 ESR1_ant: Eskiler silindi, yeniler üretiliyor...
📦 MAPK1: Eskiler silindi, yeniler üretiliyor...
📦 PKM2: Eskiler silindi, yeniler üretiliyor...
📦 ESR1_ago: Eskiler silindi, yeniler üretiliyor...
📦 GBA: Eskiler silindi, yeniler üretiliyor...
📦 MTORC1: Eskiler silindi, yeniler üretiliyor...
📦 benchmark_results_litpcba_ssd: Eskiler silindi, yeniler üretiliyor...
📦 ADRB2: Eskiler silindi, yeniler üretiliyor...
📦 KAT2A: Eskiler silindi, yeniler üretiliyor...
📦 PPARG: Eskiler silindi, yeniler üretiliyor...
📦 VDR: Eskiler silindi, yeniler üretiliyor...
📦 TP53: Eskiler silindi, yeniler üretiliyor...
📦 IDH1: Eskiler silindi, yeniler üretiliyor...

✨ LIT-PCBA klasörü tamamen temizlendi ve OpenBabel ile onarıldı!


In [9]:
import h5py
# Bir tane örnek target seçelim (Mesela PKM2)
target_name = "4g1n"
h5_path = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/lit_pcba_esm2_embeddings.h5"

with h5py.File(h5_path, 'r') as f:
    emb = f[target_name][:]
    print(f"📊 {target_name} için ESM-2 Embedding Uzunluğu: {len(emb)}")

# Şimdi PDB'deki toplam atom/kalıntı sayısına bakalım
# Eğer embedding uzunluğu (örn: 531), PDB'deki kalıntı sayısından (örn: 450) fazlaysa,
# bu embedding'ler TÜM PROTEİN DİZİSİ (Full Sequence) için oluşturulmuştur.

📊 4g1n için ESM-2 Embedding Uzunluğu: 518


In [10]:
import h5py

h5_path = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/lit_pcba_esm2_embeddings.h5"

with h5py.File(h5_path, 'r') as f:
    # Tüm anahtarları al
    all_keys = list(f.keys())
    print(f"🔑 Toplam Anahtar Sayısı: {len(all_keys)}")
    print(f"📝 İlk 10 Anahtar: {all_keys[:10]}")
    
    # PKM2 içeren anahtarları filtrele
    pkm2_matches = [k for k in all_keys if "PKM2" in k.upper()]
    print(f"🔍 PKM2 ile ilgili bulunanlar: {pkm2_matches}")
    
    if pkm2_matches:
        # İlk eşleşen anahtarın boyutuna bakalım
        target = pkm2_matches[0]
        emb_shape = f[target].shape
        print(f"\n🎯 Örnek Target: {target}")
        print(f"📊 Embedding Boyutu: {emb_shape}")

🔑 Toplam Anahtar Sayısı: 129
📝 İlk 10 Anahtar: ['1fap', '1l2i', '1nsg', '1pme', '1xp1', '1xqc', '1zgy', '2ayr', '2b1v', '2b1z']
🔍 PKM2 ile ilgili bulunanlar: []


In [11]:
import h5py
import os
import numpy as np
from rdkit import Chem
from collections import defaultdict

# --- 1. AYARLAR (Kontrol etmek istediğin hedef) ---
target_name = "PKM2"
pdb_id = "4g1n" # Küçük harf (Dosya adı için)
h5_path = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/lit_pcba_esm2_embeddings.h5"
base_path = f"/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/{target_name}"

prot_path = os.path.join(base_path, f"{pdb_id}_protein.pdb")
lig_path = os.path.join(base_path, f"{pdb_id}_ligand.sdf")

print(f"🚀 {target_name} ({pdb_id}) için Teşhis Başlatıldı...\n")

# --- 2. PDB & SDF OKUMA TESTİ ---
prot = Chem.MolFromPDBFile(prot_path, sanitize=False)
suppl = Chem.SDMolSupplier(lig_path, sanitize=False)
ref_mol = suppl[0] if len(suppl) > 0 else None

if prot is None or ref_mol is None:
    print(f"❌ HATA: Dosya okunamıyor! Protein: {prot is not None}, Ligand: {ref_mol is not None}")
else:
    print(f"✅ Dosyalar başarıyla RDKit'e yüklendi.")
    print(f"   Protein Atom Sayısı: {prot.GetNumAtoms()}")
    print(f"   Ligand Atom Sayısı: {ref_mol.GetNumAtoms()}")

# --- 3. POCKET OLUŞUYOR MU? (HİZALAMA VE İSİM KONTROLÜ) ---
center = np.mean(ref_mol.GetConformer().GetPositions(), axis=0)
conf_prot = prot.GetConformer()
res_in_pocket = set()
total_atoms_in_12A = 0

for atom in prot.GetAtoms():
    pos = conf_prot.GetAtomPosition(atom.GetIdx())
    dist = np.sqrt((pos.x - center[0])**2 + (pos.y - center[1])**2 + (pos.z - center[2])**2)
    if dist <= 12.0:
        total_atoms_in_12A += 1
        info = atom.GetPDBResidueInfo()
        if info:
            res_name = info.GetResidueName().strip()
            if res_name != 'UNL':
                res_in_pocket.add((info.GetResidueNumber(), res_name))

print("-" * 30)
print(f"📏 12.0 Å Mesafe Analizi:")
print(f"   Bulunan Toplam Atom: {total_atoms_in_12A}")
print(f"   Tanımlanan Benzersiz Kalıntı Sayısı: {len(res_in_pocket)}")
if len(res_in_pocket) > 5:
    print(f"✅ CEP OLUŞUYOR: Pocket içinde {len(res_in_pocket)} geçerli amino asit bulundu.")
else:
    print(f"❌ CEP HATASI: Atomlar çok uzak veya amino asit isimleri hala geçersiz!")

# --- 4. ESM-2 EMBEDDING SENKRONİZASYONU ---
print("-" * 30)
try:
    with h5py.File(h5_path, 'r') as f:
        pdb_id_upper = pdb_id.upper()
        if pdb_id_upper in f:
            emb = f[pdb_id_upper][:]
            # PDB'deki toplam kalıntı sayısını bulalım
            unique_res_in_all_pdb = set()
            for atom in prot.GetAtoms():
                info = atom.GetPDBResidueInfo()
                if info:
                    unique_res_in_all_pdb.add((info.GetResidueNumber(), info.GetChainId()))
            
            pdb_res_count = len(unique_res_in_all_pdb)
            emb_len = len(emb)
            
            print(f"🧬 ESM-2 Senkronizasyon Analizi:")
            print(f"   PDB'deki Kalıntı Sayısı: {pdb_res_count}")
            print(f"   H5'teki Embedding Uzunluğu: {emb_len}")
            
            if pdb_res_count == emb_len:
                print(f"✅ TAM EŞLEŞME: PDB ve ESM-2 verileri birebir uyumlu.")
            else:
                print(f"⚠️ DİKKAT: Sayılar tutmuyor! Fark: {abs(pdb_res_count - emb_len)}")
                print(f"   (Eğer H5 dosyası Mol2'ye göre hazırlandıysa bu fark normal olabilir.)")
        else:
            print(f"❌ H5 HATASI: {pdb_id_upper} anahtarı H5 dosyasında yok!")
except Exception as e:
    print(f"❌ H5 OKUMA HATASI: {e}")

🚀 PKM2 (4g1n) için Teşhis Başlatıldı...

✅ Dosyalar başarıyla RDKit'e yüklendi.
   Protein Atom Sayısı: 8016
   Ligand Atom Sayısı: 56
------------------------------
📏 12.0 Å Mesafe Analizi:
   Bulunan Toplam Atom: 364
   Tanımlanan Benzersiz Kalıntı Sayısı: 43
✅ CEP OLUŞUYOR: Pocket içinde 43 geçerli amino asit bulundu.
------------------------------
❌ H5 HATASI: 4G1N anahtarı H5 dosyasında yok!


In [13]:
import h5py
h5_new = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/LIT-PCBA/lit_pcba_esm2_embeddings.h5"

with h5py.File(h5_new, 'r') as f:
    if "4g1n" in f:
        print(f"✅ ZAFER! 4G1N artık H5 dosyasında mevcut. Uzunluk: {len(f['4g1n'])}")
    else:
        print("❌ Hala yok. Embedding scriptinin çıktılarını (output) kontrol etmelisin.")

✅ ZAFER! 4G1N artık H5 dosyasında mevcut. Uzunluk: 518


In [1]:
import torch
import sys
import os

# Yolları ekleyelim
sys.path.append("/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/Dataset_preparation")

from PLANET_datautils_esm2_vector import ProLigDataset

PKL_PATH = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/pdbbind2020/index/valid_esm2.pkl" 

def test_coordinates():
    if not os.path.exists(PKL_PATH):
        print(f"❌ HATA: {PKL_PATH} dosyası bulunamadı. Lütfen script içindeki yolu düzeltin.")
        return

    print("📂 Dataset yükleniyor...")
    # Batch size 2 olsun ki collate fonksiyonu çalışsın
    dataset = ProLigDataset(PKL_PATH, batch_size=2, shuffle=False)
    
    print("⚡ İlk batch çekiliyor...")
    batch = dataset[0] # __getitem__ çağrılır -> tensorize çalışır

    if batch is None:
        print("❌ Batch boş döndü! (Dosya okuma hatası olabilir)")
        return

    res_batch, mol_batch, targets = batch

    print("-" * 30)
    print(f"📦 Mol Batch Eleman Sayısı: {len(mol_batch)}")
    print("-" * 30)

    # Beklenen sayı: 6 (Eski kodda 5 idi)
    if len(mol_batch) == 6:
        print("✅ BAŞARILI: Mol Batch 6 elemanlı! (Koordinatlar eklenmiş)")
        
        lig_coords = mol_batch[5] # 6. eleman (index 5)
        
        print(f"📊 Koordinat Shape: {lig_coords.shape}")
        print(f"🧪 Veri Tipi: {lig_coords.dtype}")
        
        # İçerik Kontrolü
        if lig_coords.shape[1] != 3:
             print("❌ HATA: Koordinatların 2. boyutu 3 olmalı (x,y,z)!")
        
        is_all_zero = (lig_coords == 0).all()
        if is_all_zero:
            print("⚠️ UYARI: Koordinatların HEPSİ SIFIR. (RDKit Conformer üretmemiş veya 3D veri yok)")
        else:
            print("✅ Veri Dolu: Koordinatlar sıfırdan farklı değerler içeriyor.")
            print(f"📍 İlk 3 Atomun Konumu:\n{lig_coords[:3]}")
            
    else:
        print(f"❌ BAŞARISIZ: Mol Batch hala {len(mol_batch)} elemanlı. `chemutils_esm2.py` güncellenmemiş veya import cache sorunu var.")

if __name__ == "__main__":
    test_coordinates()

📂 Dataset yükleniyor...
⚡ İlk batch çekiliyor...
------------------------------
📦 Mol Batch Eleman Sayısı: 6
------------------------------
✅ BAŞARILI: Mol Batch 6 elemanlı! (Koordinatlar eklenmiş)
📊 Koordinat Shape: torch.Size([32, 3])
🧪 Veri Tipi: torch.float32
✅ Veri Dolu: Koordinatlar sıfırdan farklı değerler içeriyor.
📍 İlk 3 Atomun Konumu:
tensor([[ -5.4250,  22.2210, -29.5840],
        [ -8.0940,  25.8970, -38.9430],
        [ -7.4660,  28.3960, -38.0110]])
